In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# ========== CHANGE THESE TWO LINES ==========
CSV_PATH = r"PATH\TO\your_cancer_dataset.csv"
TARGET_COL = "diagnosis"   # e.g., "diagnosis" or "target" (change if needed)
# ===========================================

df = pd.read_csv(r"data.csv")

# Drop common junk columns like "Unnamed: 32"
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", regex=True)]

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# Split features/target
# Split features/target
TARGET_COL = "diagnosis"

X = df.drop(columns=[TARGET_COL]).copy()

# Drop ID if present
if "id" in X.columns:
    X = X.drop(columns=["id"])

y_raw = df[TARGET_COL].astype(str)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y_raw)

print("Target mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
# Typically: {'B': 0, 'M': 1}

# Train/test split (stratify keeps class balance)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

import numpy as np
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Class balance (train):", np.bincount(y_train))
print("Class balance (test):", np.bincount(y_test))


Shape: (569, 32)
Columns: ['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']
Target mapping: {'B': np.int64(0), 'M': np.int64(1)}
Train shape: (455, 30) Test shape: (114, 30)
Class balance (train): [285 170]
Class balance (test): [72 42]


In [6]:
import numpy as np

from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef
)

knn = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(
        n_neighbors=5,      # default baseline
        weights="distance", # often better than uniform
        metric="minkowski", # Euclidean (p=2) by default
        p=2
    ))
])

# Train
knn.fit(X_train, y_train)

# Predict
y_pred = knn.predict(X_test)

# AUC: KNN supports predict_proba
y_proba = knn.predict_proba(X_test)[:, 1]

# Metrics
metrics_knn = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "MCC": matthews_corrcoef(y_test, y_pred),
}

print("\n=== Model 3: KNN Metrics ===")
for k, v in metrics_knn.items():
    print(f"{k:10s}: {v:.6f}")



=== Model 3: KNN Metrics ===
Accuracy  : 0.956140
AUC       : 0.982474
Precision : 0.974359
Recall    : 0.904762
F1        : 0.938272
MCC       : 0.905824
